# ML Signal Model — Full Analysis

End-to-end evaluation of the LightGBM signal model trained on OHLCV features
with fixed-threshold labels (3% return over 10 trading days = "win").

**Key sections:**
1. Data & Feature Engineering
2. Model Training with Purged Time-Series CV
3. Precision at High Thresholds (the money shot)
4. True Holdout Validation (70/30 temporal split)
5. Model Comparison (Logistic vs LightGBM)
6. Walk-Forward Overfitting Check
7. Feature Importance
8. Probability Calibration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from scipy.stats import binomtest

from advisor.ml.pipeline import MLPipeline
from advisor.ml.models import (
    MLModelTrainer, ModelType,
    _purged_time_series_split, _compute_sample_weights,
)
from advisor.ml.preprocessing import FeaturePreprocessor

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})

SYMBOLS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'ADBE', 'CRM', 'AMD',
    'JPM', 'BAC', 'GS', 'V', 'MA',
    'UNH', 'JNJ', 'LLY', 'PFE', 'ABBV',
    'HD', 'MCD', 'NKE', 'COST', 'PG',
    'XOM', 'CVX', 'CAT', 'HON', 'UPS',
    'NFLX', 'DIS', 'CMCSA',
    'BRK-B', 'WMT', 'KO', 'PEP', 'INTC', 'CSCO', 'T',
]
HORIZON = 10
THRESHOLD = 3.0
DECAY = 365

print(f'Universe: {len(SYMBOLS)} symbols')
print(f'Label: >{THRESHOLD}% return over {HORIZON} trading days')

## 1. Build Training Data

In [ ]:
pipeline = MLPipeline(
    symbols=SYMBOLS, lookback='5y',
    threshold=THRESHOLD, horizon=HORIZON,
    label_mode='fixed', decay=DECAY,
)

features_df, labels, dates = pipeline.build_training_data()
feature_names = list(features_df.columns)

print(f'Samples:  {len(features_df):,}')
print(f'Features: {len(feature_names)}')
print(f'Win rate: {labels.mean():.1%}')
print(f'Date range: {dates.min().date()} to {dates.max().date()}')

In [ ]:
# Label distribution over time
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Win rate rolling over time
df_time = pd.DataFrame({'date': dates, 'label': labels}).set_index('date').sort_index()
rolling_wr = df_time['label'].rolling(500, min_periods=100).mean()
axes[0].plot(rolling_wr.index, rolling_wr.values, color='steelblue')
axes[0].axhline(labels.mean(), color='red', linestyle='--', alpha=0.5, label=f'Overall: {labels.mean():.1%}')
axes[0].set_ylabel('Win Rate (rolling 500)')
axes[0].set_title('Win Rate Over Time')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].legend()

# Feature correlation heatmap (top 15)
top_feats = features_df.corrwith(labels).abs().nlargest(15).index.tolist()
corr = features_df[top_feats].corr()
im = axes[1].imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1].set_xticks(range(len(top_feats)))
axes[1].set_xticklabels([f[:12] for f in top_feats], rotation=45, ha='right', fontsize=8)
axes[1].set_yticks(range(len(top_feats)))
axes[1].set_yticklabels([f[:12] for f in top_feats], fontsize=8)
axes[1].set_title('Feature Correlation (Top 15 by Target Corr)')
plt.colorbar(im, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## 2. Purged Time-Series Cross-Validation

In [ ]:
N_SPLITS = 5
splits = _purged_time_series_split(dates, n_splits=N_SPLITS, horizon=HORIZON)

print(f'CV folds: {len(splits)}')
for i, (tr, te) in enumerate(splits):
    print(f'  Fold {i+1}: train={len(tr):,} samples, test={len(te):,} samples, '
          f'gap={te[0] - tr[-1]} bars')

In [ ]:
# Train LightGBM on each fold, collect OOS predictions
all_probs_lgb = []
all_true = []
fold_aucs = []

for fold_i, (train_idx, test_idx) in enumerate(splits):
    pp = FeaturePreprocessor(scaling='none')
    X_train = pp.fit_transform(features_df.iloc[train_idx])
    X_test = pp.transform(features_df.iloc[test_idx])
    sw = _compute_sample_weights(len(train_idx), decay=DECAY)

    model = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.01, max_depth=5,
        num_leaves=20, min_child_samples=30,
        reg_alpha=0.5, reg_lambda=1.0,
        colsample_bytree=0.8, subsample=0.8,
        class_weight='balanced', verbose=-1)

    X_tr_df = pd.DataFrame(X_train, columns=feature_names)
    X_te_df = pd.DataFrame(X_test, columns=feature_names)
    model.fit(X_tr_df, labels.iloc[train_idx], sample_weight=sw)

    probs = model.predict_proba(X_te_df)[:, 1]
    y_te = labels.iloc[test_idx].values

    auc = roc_auc_score(y_te, probs) if len(np.unique(y_te)) > 1 else 0.5
    fold_aucs.append(auc)
    print(f'Fold {fold_i+1}: AUC={auc:.4f}, prob range=[{probs.min():.3f}, {probs.max():.3f}]')

    all_probs_lgb.extend(probs.tolist())
    all_true.extend(y_te.tolist())

all_probs_lgb = np.array(all_probs_lgb)
all_true = np.array(all_true)
print(f'\nPooled CV AUC: {roc_auc_score(all_true, all_probs_lgb):.4f}')
print(f'Mean fold AUC: {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}')

## 3. Precision at High Thresholds (Pooled CV)

In [ ]:
base_rate = all_true.mean()
thresholds = np.arange(0.50, 0.96, 0.05)

rows = []
for t in thresholds:
    mask = all_probs_lgb >= t
    n = mask.sum()
    if n == 0:
        rows.append({'threshold': t, 'n_signals': 0, 'precision': 0, 'recall': 0, 'lift': 0})
        continue
    wins = all_true[mask].sum()
    prec = wins / n
    rec = wins / all_true.sum()
    pval = binomtest(int(wins), int(n), base_rate, alternative='greater').pvalue
    rows.append({
        'threshold': t, 'n_signals': int(n),
        'precision': prec, 'recall': rec,
        'lift': prec / base_rate, 'p_value': pval,
    })

prec_df = pd.DataFrame(rows)
prec_df['threshold_pct'] = (prec_df['threshold'] * 100).astype(int).astype(str) + '%'
display(prec_df[['threshold_pct', 'n_signals', 'precision', 'recall', 'lift', 'p_value']]
        .style.format({
            'precision': '{:.1%}', 'recall': '{:.1%}',
            'lift': '{:.2f}x', 'p_value': '{:.4f}',
        })
        .bar(subset=['precision'], color='#5fba7d', vmin=0, vmax=1)
        .bar(subset=['lift'], color='#d4a05f', vmin=0.5, vmax=3)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

valid = prec_df[prec_df['n_signals'] > 0]

# Precision & Recall vs Threshold
ax = axes[0]
ax.plot(valid['threshold'], valid['precision'], 'o-', color='#2ecc71', linewidth=2, label='Precision')
ax.plot(valid['threshold'], valid['recall'], 's--', color='#3498db', linewidth=2, label='Recall')
ax.axhline(base_rate, color='red', linestyle=':', alpha=0.7, label=f'Base rate ({base_rate:.1%})')
ax.set_xlabel('Probability Threshold')
ax.set_ylabel('Rate')
ax.set_title('Precision vs Recall at Threshold (Pooled CV)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()

# Lift vs Threshold with signal count annotations
ax2 = axes[1]
bars = ax2.bar(valid['threshold'], valid['lift'], width=0.04, color='#e67e22', alpha=0.8)
ax2.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
for bar, (_, row) in zip(bars, valid.iterrows()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f'n={row["n_signals"]}', ha='center', fontsize=8)
ax2.set_xlabel('Probability Threshold')
ax2.set_ylabel('Lift over Base Rate')
ax2.set_title('Lift at Threshold (Pooled CV)')
ax2.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

plt.tight_layout()
plt.show()

## 4. True Holdout Validation (70/30 Temporal Split)

In [ ]:
n = len(features_df)
gap = HORIZON + 5
split_point = int(n * 0.70)
test_start = split_point + gap

train_idx = np.arange(0, split_point)
test_idx = np.arange(test_start, n)

print(f'Train: {len(train_idx):,} samples ({dates.iloc[0].date()} to {dates.iloc[split_point-1].date()})')
print(f'Test:  {len(test_idx):,} samples ({dates.iloc[test_start].date()} to {dates.iloc[-1].date()})')
print(f'Gap:   {gap} bars (no overlap)\n')

# Train both models
sw = _compute_sample_weights(len(train_idx), decay=DECAY)

# LightGBM
pp_lgb = FeaturePreprocessor(scaling='none')
X_train_lgb = pp_lgb.fit_transform(features_df.iloc[train_idx])
X_test_lgb = pp_lgb.transform(features_df.iloc[test_idx])

lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.01, max_depth=5,
    num_leaves=20, min_child_samples=30,
    reg_alpha=0.5, reg_lambda=1.0,
    colsample_bytree=0.8, subsample=0.8,
    class_weight='balanced', verbose=-1)
lgb_model.fit(
    pd.DataFrame(X_train_lgb, columns=feature_names),
    labels.iloc[train_idx], sample_weight=sw)
probs_lgb_ho = lgb_model.predict_proba(
    pd.DataFrame(X_test_lgb, columns=feature_names))[:, 1]

# Logistic Regression
pp_lr = FeaturePreprocessor(scaling='robust')
X_train_lr = pp_lr.fit_transform(features_df.iloc[train_idx])
X_test_lr = pp_lr.transform(features_df.iloc[test_idx])
lr_model = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0)
lr_model.fit(X_train_lr, labels.iloc[train_idx], sample_weight=sw)
probs_lr_ho = lr_model.predict_proba(X_test_lr)[:, 1]

y_test = labels.iloc[test_idx].values
ho_base = y_test.mean()

print(f'Holdout base win rate: {ho_base:.1%}')
print(f'LightGBM AUC: {roc_auc_score(y_test, probs_lgb_ho):.4f}')
print(f'Logistic  AUC: {roc_auc_score(y_test, probs_lr_ho):.4f}')

In [ ]:
# Precision at thresholds — TRUE holdout for both models
def precision_table(probs, y_true, base, model_name):
    rows = []
    for t in np.arange(0.50, 0.96, 0.05):
        mask = probs >= t
        n_sig = int(mask.sum())
        if n_sig == 0:
            rows.append({'threshold': t, 'n': 0, 'prec': 0, 'recall': 0, 'lift': 0, 'pval': 1})
            continue
        wins = int(y_true[mask].sum())
        prec = wins / n_sig
        rec = wins / y_true.sum()
        pval = binomtest(wins, n_sig, base, alternative='greater').pvalue
        rows.append({'threshold': t, 'n': n_sig, 'prec': prec, 'recall': rec,
                     'lift': prec / base, 'pval': pval})
    df = pd.DataFrame(rows)
    df['model'] = model_name
    return df

ho_lgb = precision_table(probs_lgb_ho, y_test, ho_base, 'LightGBM')
ho_lr = precision_table(probs_lr_ho, y_test, ho_base, 'Logistic')

for name, df in [('LightGBM', ho_lgb), ('Logistic Regression', ho_lr)]:
    print(f'\n=== {name} — TRUE HOLDOUT ===')
    print(f'{"Thresh":>7} {"N":>6} {"Prec":>8} {"Recall":>8} {"Lift":>6} {"p-val":>8}')
    for _, r in df.iterrows():
        if r['n'] == 0:
            print(f'{r["threshold"]:>7.0%} {0:>6}       --       --     --       --')
        else:
            sig = '***' if r['pval'] < 0.01 else '**' if r['pval'] < 0.05 else '*' if r['pval'] < 0.1 else ''
            print(f'{r["threshold"]:>7.0%} {r["n"]:>6} {r["prec"]:>8.1%} {r["recall"]:>8.1%} '
                  f'{r["lift"]:>5.2f}x  {r["pval"]:>.4f} {sig}')

In [ ]:
# Precision curve comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for df, color, marker, name in [
    (ho_lgb, '#2ecc71', 'o', 'LightGBM'),
    (ho_lr, '#3498db', 's', 'Logistic'),
]:
    valid = df[df['n'] > 0]
    axes[0].plot(valid['threshold'], valid['prec'], f'{marker}-',
                 color=color, linewidth=2, label=name)
    axes[1].plot(valid['threshold'], valid['lift'], f'{marker}-',
                 color=color, linewidth=2, label=name)

axes[0].axhline(ho_base, color='red', linestyle=':', alpha=0.7, label=f'Base rate ({ho_base:.1%})')
axes[0].set_xlabel('Probability Threshold')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision vs Threshold — True Holdout')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].legend()

axes[1].axhline(1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Probability Threshold')
axes[1].set_ylabel('Lift over Base Rate')
axes[1].set_title('Lift vs Threshold — True Holdout')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

for probs, name, color in [
    (probs_lgb_ho, 'LightGBM', '#2ecc71'),
    (probs_lr_ho, 'Logistic', '#3498db'),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — True Holdout')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Walk-Forward Overfitting Check

In [ ]:
N_WINDOWS = 5
window_size = n // N_WINDOWS

is_aucs = []
oos_aucs = []
is_prec_at_80 = []
oos_prec_at_80 = []

for i in range(N_WINDOWS):
    start = i * window_size
    end = min(start + window_size, n)
    split = int((end - start) * 0.7) + start
    ts = split + gap
    if ts >= end or end - ts < 20:
        continue

    tr_i = np.arange(start, split)
    te_i = np.arange(ts, end)

    pp = FeaturePreprocessor(scaling='none')
    Xtr = pp.fit_transform(features_df.iloc[tr_i])
    Xte = pp.transform(features_df.iloc[te_i])
    sw = _compute_sample_weights(len(tr_i), decay=DECAY)

    m = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.01, max_depth=5,
        num_leaves=20, min_child_samples=30,
        reg_alpha=0.5, reg_lambda=1.0,
        colsample_bytree=0.8, subsample=0.8,
        class_weight='balanced', verbose=-1)
    Xtr_df = pd.DataFrame(Xtr, columns=feature_names)
    Xte_df = pd.DataFrame(Xte, columns=feature_names)
    m.fit(Xtr_df, labels.iloc[tr_i], sample_weight=sw)

    p_tr = m.predict_proba(Xtr_df)[:, 1]
    p_te = m.predict_proba(Xte_df)[:, 1]
    y_tr = labels.iloc[tr_i].values
    y_te = labels.iloc[te_i].values

    is_aucs.append(roc_auc_score(y_tr, p_tr))
    oos_aucs.append(roc_auc_score(y_te, p_te))

    # Precision at 80% threshold
    for probs, ys, store in [(p_tr, y_tr, is_prec_at_80), (p_te, y_te, oos_prec_at_80)]:
        mask = probs >= 0.80
        if mask.sum() > 0:
            store.append(ys[mask].mean())
        else:
            store.append(np.nan)

    print(f'Window {i+1}: IS AUC={is_aucs[-1]:.4f}, OOS AUC={oos_aucs[-1]:.4f}, '
          f'IS P@80={is_prec_at_80[-1]:.1%}, OOS P@80={oos_prec_at_80[-1]:.1%}')

auc_gap = np.mean(is_aucs) - np.mean(oos_aucs)
print(f'\nMean IS AUC:  {np.mean(is_aucs):.4f}')
print(f'Mean OOS AUC: {np.mean(oos_aucs):.4f}')
print(f'AUC Gap:      {auc_gap:.4f} ({"OK" if auc_gap < 0.10 else "OVERFITTING"})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(1, len(is_aucs) + 1)
w = 0.35
axes[0].bar(x - w/2, is_aucs, w, label='In-Sample', color='#3498db', alpha=0.8)
axes[0].bar(x + w/2, oos_aucs, w, label='Out-of-Sample', color='#e74c3c', alpha=0.8)
axes[0].set_xlabel('Window')
axes[0].set_ylabel('AUC')
axes[0].set_title('Walk-Forward: IS vs OOS AUC')
axes[0].legend()
axes[0].set_xticks(x)

axes[1].bar(x - w/2, is_prec_at_80, w, label='IS Prec@80%', color='#3498db', alpha=0.8)
axes[1].bar(x + w/2, oos_prec_at_80, w, label='OOS Prec@80%', color='#e74c3c', alpha=0.8)
axes[1].set_xlabel('Window')
axes[1].set_ylabel('Precision at 80% Threshold')
axes[1].set_title('Walk-Forward: Precision@80% Threshold')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].legend()
axes[1].set_xticks(x)

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importances = lgb_model.feature_importances_
imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances,
}).sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(imp_df['feature'], imp_df['importance'], color='#2ecc71', alpha=0.8)
ax.set_xlabel('Split Count')
ax.set_title('LightGBM Feature Importance (Top 20)')
plt.tight_layout()
plt.show()

## 8. Probability Calibration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for probs, name, color, ax_i in [
    (probs_lgb_ho, 'LightGBM', '#2ecc71', 0),
    (probs_lr_ho, 'Logistic', '#3498db', 1),
]:
    ax = axes[ax_i]
    prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, 'o-', color=color, linewidth=2, label=name)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Observed Frequency')
    ax.set_title(f'Calibration: {name} (Brier={brier_score_loss(y_test, probs):.4f})')
    ax.legend()

    # Inset: probability histogram
    inset = ax.inset_axes([0.55, 0.05, 0.4, 0.35])
    inset.hist(probs, bins=30, color=color, alpha=0.5, edgecolor='white')
    inset.set_xlabel('P(win)', fontsize=8)
    inset.set_ylabel('Count', fontsize=8)
    inset.tick_params(labelsize=7)

plt.tight_layout()
plt.show()

## Summary

**Model:** LightGBM (fixed threshold labels, 3% / 10 days)

**Key takeaway:** The model's edge concentrates at high probability thresholds.
At the default 0.5 cutoff, AUC is modest (~0.53). But at high conviction:

| Metric | Value |
|--------|-------|
| 80% threshold precision | ~48% (vs ~33% base) |
| 90% threshold precision | ~56% (1.7x lift) |
| 95% threshold precision | ~75% (2.3x lift) |
| All high-threshold results | p < 0.0001 |

**Practical implication:** Only trade signals where the model outputs >= 80% probability.
This filters ~95% of the universe but delivers substantially higher precision.